# Column profile QC before cleaning

This notebook reads only raw profiles from `profiles_before_clean`. It does not interpolate, delete, or overwrite any column.

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
from IPython.display import display

RESULTS_ROOT = Path('/mnt/newStor/paros/paros_WORK/xintong/results/column_analysis')
QC_ROOT = RESULTS_ROOT / 'column_profile_qc_before_clean'
QC_ROOT.mkdir(parents=True, exist_ok=True)
METRICS = ('md', 'fa', 'ad', 'rd')
DEPTHS = 21
print('Reading:', RESULTS_ROOT)
print('Writing QC tables to:', QC_ROOT)

Reading: /mnt/newStor/paros/paros_WORK/xintong/results/column_analysis
Writing QC tables to: /mnt/newStor/paros/paros_WORK/xintong/results/column_analysis/column_profile_qc_before_clean


: 

## 1. Load raw profiles

Each CSV is treated as `subject × hemisphere × region × metric`; rows are columns/trajectories and the 21 columns are cortical depths.

In [ ]:
rows = []
pattern = re.compile(r'^(?P<subject>D\d+)_(?P<hemi>lh|rh)_(?P<region>.+)_(?P<metric>md|fa|ad|rd)\.csv$')
for profile_dir in sorted(RESULTS_ROOT.glob('*/profiles_before_clean')):
    for path in sorted(profile_dir.glob('*.csv')):
        m = pattern.match(path.name)
        if not m:
            continue
        values = np.loadtxt(path, delimiter=',', ndmin=2)
        if values.shape[1] != DEPTHS:
            print('SKIP unexpected shape:', path, values.shape)
            continue
        for column_id, vector in enumerate(values):
            rows.append({**m.groupdict(), 'column_id': column_id, 'values': vector, 'path': str(path)})
profiles = pd.DataFrame(rows)
if profiles.empty:
    raise RuntimeError('No profiles_before_clean files found.')
print('Subjects:', profiles.subject.nunique())
print('Profile rows:', len(profiles))
print('Metrics:', sorted(profiles.metric.unique()))

## 2. Classify bad points

A bad point is classified separately as exact zero, NaN, or Inf. The summary does not combine these into a cleaning decision.

In [ ]:
def point_type(x):
    if np.isnan(x): return 'nan'
    if np.isinf(x): return 'inf'
    if x == 0: return 'exact_zero'
    return 'valid'

profiles['types'] = profiles['values'].map(lambda v: np.array([point_type(x) for x in v]))
profiles['bad_mask'] = profiles['types'].map(lambda v: v != 'valid')
profiles['bad_count'] = profiles['bad_mask'].map(int.__len__)
profiles['bad_count'] = profiles['bad_mask'].map(np.count_nonzero)
profiles['has_bad'] = profiles['bad_count'] > 0
profiles['runs'] = profiles['bad_mask'].map(lambda v: np.diff(np.r_[False, v, False].astype(int)))
profiles['max_consecutive_bad'] = profiles['runs'].map(lambda d: int(max([b-a for a,b in zip(np.where(d==1)[0], np.where(d==-1)[0])] or [0])))
profiles['bad_type_counts'] = profiles['types'].map(lambda v: {k: int(np.sum(v == k)) for k in ('exact_zero','nan','inf')})

## 3. Subject/metric summary

This produces total columns, columns with bad points, bad-point totals, one/two/three-or-more bad points, and continuity measures.

In [ ]:
def summarize_group(g):
    bad = g.bad_mask.to_numpy()
    types = np.concatenate(g.types.to_numpy())
    return pd.Series({
        'total_columns': len(g),
        'bad_points': int(bad.sum()),
        'total_sampling_points': int(bad.size),
        'bad_point_percent': 100 * bad.mean(),
        'columns_with_bad_points': int(g.has_bad.sum()),
        'columns_with_bad_percent': 100 * g.has_bad.mean(),
        'one_bad_point': int((g.bad_count == 1).sum()),
        'two_bad_points': int((g.bad_count == 2).sum()),
        'three_or_more_bad_points': int((g.bad_count >= 3).sum()),
        'columns_with_consecutive_bad': int((g.max_consecutive_bad >= 2).sum()),
        'max_consecutive_bad_points': int(g.max_consecutive_bad.max()),
        'exact_zero_points': int((types == 'exact_zero').sum()),
        'nan_points': int((types == 'nan').sum()),
        'inf_points': int((types == 'inf').sum()),
    })
summary = profiles.groupby(['subject','metric'], as_index=False).apply(summarize_group, include_groups=False).reset_index()
summary = summary.drop(columns=['level_2'], errors='ignore')
summary.to_csv(QC_ROOT / 'column_profile_summary_by_subject_metric.csv', index=False)
display(summary)

## 4. Depth distribution

Depth 0 and depth 20 are reported explicitly; all other depths are labelled `middle`.

In [ ]:
depth_rows = []
for _, r in profiles.iterrows():
    for depth, bad in enumerate(r.bad_mask):
        if bad:
            depth_rows.append({'subject':r.subject,'metric':r.metric,'hemisphere':r.hemi,'region':r.region,'column_id':r.column_id,'depth':depth,'depth_group':'depth_0' if depth==0 else 'depth_20' if depth==20 else 'middle','bad_type':r.types[depth]})
depth_bad = pd.DataFrame(depth_rows)
if depth_bad.empty:
    depth_summary = pd.DataFrame(columns=['metric','depth','bad_point_count','bad_point_percentage'])
else:
    depth_summary = depth_bad.groupby(['metric','depth'], as_index=False).size().rename(columns={'size':'bad_point_count'})
    depth_summary['bad_point_percentage'] = depth_summary.groupby('metric')['bad_point_count'].transform(lambda x: 100*x/x.sum())
depth_summary.to_csv(QC_ROOT / 'depth_distribution.csv', index=False)
display(depth_summary)

## 5. Hemisphere/region and cross-metric overlap

The first table preserves subject, hemisphere, and region. The second checks whether the same column/depth is bad across multiple metrics.

In [ ]:
region_summary = profiles.groupby(['subject','hemisphere','region','metric'], as_index=False).apply(summarize_group, include_groups=False).reset_index().drop(columns=['level_2'], errors='ignore')
region_summary.to_csv(QC_ROOT / 'column_profile_summary_by_region.csv', index=False)

wide = profiles.pivot_table(index=['subject','hemi','region','column_id'], columns='metric', values='bad_mask', aggfunc='first')
overlap = []
for idx, r in wide.iterrows():
    present = [m for m in METRICS if m in r.index and isinstance(r[m], np.ndarray)]
    if len(present) < 2: continue
    masks = [r[m] for m in present]
    overlap.append({'subject':idx[0],'hemisphere':idx[1],'region':idx[2],'column_id':idx[3],'metrics_present':','.join(present),'bad_in_all_metrics':int(np.logical_and.reduce(masks).sum()),'bad_in_any_metric':int(np.logical_or.reduce(masks).sum())})
overlap = pd.DataFrame(overlap)
overlap.to_csv(QC_ROOT / 'cross_metric_column_overlap.csv', index=False)
display(region_summary.head())
display(overlap.head())
print('Saved raw-profile QC tables:', QC_ROOT)